In [10]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

print("device_count:", torch.cuda.device_count())
print("current_device:", torch.cuda.current_device())
print("device_name(0):", torch.cuda.get_device_name(0))
print("device_string:", torch.device("cuda:0"))

2.3.1
CUDA available: True
CUDA version: 12.1
GPU: NVIDIA GeForce GTX 1650
device_count: 1
current_device: 0
device_name(0): NVIDIA GeForce GTX 1650
device_string: cuda:0


In [57]:
import pandas as pd

s_data = pd.read_parquet('all_samples.parquet')
ecg = s_data.loc[:, ['ecg_raw', 'label']].copy()

bad2good_idx = ecg[ecg['label']=='bad2good']['label'].index
ecg.loc[bad2good_idx, ['label']] = 'good'

label_map = {'good':0, 'bad': 1}
ecg['label_int'] = ecg['label'].map(label_map)
data = ecg.copy()

## 数据载入：Transform, Dataset

In [58]:
from split_and_train import *

# Dataset
ds = ECGDataset(data, transform=ecg_transform)

# 用 sampler 解决不平衡
labels = data["label_int"].to_numpy()
class_counts = np.bincount(labels)                 # [good, bad]
class_weights = 1.0 / class_counts
sample_weights = class_weights[labels]
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(labels),
    replacement=True,
)

loader = DataLoader(ds, batch_size=64, sampler=sampler, num_workers=0)

batch = next(iter(loader))
print(batch["data"].shape, batch["labels"].shape)  # (64, 1, 2500) (64,)

torch.Size([64, 1, 2500]) torch.Size([64])


## 准备喂模型：切分、采样、DataLoader实例化

In [59]:
train_idx, val_idx, test_idx = get_train_val_test_ds(data)
train_ds, val_ds, test_ds = get_dataset(data, train_idx, val_idx, test_idx)
print(len(train_ds), len(val_ds), len(test_ds))

train_sampler = get_sampler(data, train_idx)
train_loader, val_loader, test_loader = get_dataloader(train_ds, val_ds, test_ds, train_sampler)

batch = next(iter(train_loader))
print(batch["data"].shape, batch["labels"].shape)
batch['labels']

21172 2353 5882
torch.Size([64, 2500]) torch.Size([64])


tensor([0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1,
        0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0,
        1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0])

## 模型、训练、评估

In [60]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ConvNet().to(device)

criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

epochs = 100
best_vloss = float("inf")
history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}

best = float("inf")
patience = 8          # 你现在这抖动幅度，8差不多
min_delta = 1e-3      # 至少要比最佳值好这么多才算“真进步”
wait = 0

In [61]:
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(train_loader, model, criterion, optimizer, device)
    val_loss, val_acc     = evaluate(val_loader, model, criterion, device)

    # 保存最好模型
    if val_loss < best - min_delta:
        best = val_loss
        wait = 0
        torch.save(model.state_dict(), "best.pt")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stop at epoch {epoch}, best val_loss={best:.4f}")
            break

    history['loss'].append(train_loss)
    history['acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch:03d}: LOSS train {train_loss:.3f} valid {val_loss:.3f}; "
          f"ACC train {train_acc:.4f} valid {val_acc:.4f}")
    

Epoch 000: LOSS train 1.149 valid 0.625; ACC train 0.7533 valid 0.8151
Epoch 001: LOSS train 0.423 valid 0.241; ACC train 0.8606 valid 0.9660
Epoch 002: LOSS train 0.385 valid 0.423; ACC train 0.8715 valid 0.9409
Epoch 003: LOSS train 0.353 valid 0.206; ACC train 0.8827 valid 0.9724
Epoch 004: LOSS train 0.326 valid 0.289; ACC train 0.8873 valid 0.9626
Epoch 005: LOSS train 0.293 valid 0.241; ACC train 0.8951 valid 0.9749
Epoch 006: LOSS train 0.249 valid 0.334; ACC train 0.9080 valid 0.9571
Epoch 007: LOSS train 0.226 valid 0.194; ACC train 0.9198 valid 0.9779
Epoch 008: LOSS train 0.187 valid 0.170; ACC train 0.9349 valid 0.9775
Epoch 009: LOSS train 0.176 valid 0.091; ACC train 0.9420 valid 0.9928
Epoch 010: LOSS train 0.162 valid 0.069; ACC train 0.9476 valid 0.9902
Epoch 011: LOSS train 0.142 valid 0.062; ACC train 0.9544 valid 0.9885
Epoch 012: LOSS train 0.144 valid 0.202; ACC train 0.9539 valid 0.9596
Epoch 013: LOSS train 0.137 valid 0.080; ACC train 0.9565 valid 0.9902
Epoch 

In [62]:
from sklearn.metrics import confusion_matrix, roc_auc_score

@torch.no_grad()
def evaluate(dataloader, model, criterion, device):
    model.eval()
    total_loss = 0.0
    n = 0
    
    # 建立两个空列表，用来收集模型所有的“罪证”
    all_y_true = []
    all_y_probs = []

    for batch in dataloader:
        x = batch["data"].to(device).float()
        if x.ndim == 2:
            x = x.unsqueeze(1)
        y = batch["labels"].to(device).float()

        logits = model(x)
        loss = criterion(logits, y)

        # 算出概率（如果是二分类且用的BCEWithLogitsLoss）
        probs = torch.sigmoid(logits) 

        bs = x.size(0)
        total_loss += loss.item() * bs
        n += bs

        # 收集数据
        all_y_true.append(y.cpu().numpy())
        all_y_probs.append(probs.cpu().numpy())

    # 压平数据，把 Chunk 们融合成一个大张量
    y_true = np.concatenate(all_y_true)
    y_probs = np.concatenate(all_y_probs)
    y_preds = (y_probs > 0.5).astype(int) # 判定阈值设为 0.5

    # 1. 计算混淆矩阵 (TN, FP, FN, TP)
    cm = confusion_matrix(y_true, y_preds, normalize='true')
    
    # 2. 计算 ROC-AUC (这才是你那 0.99999 谎言的照妖镜)
    try:
        auc = roc_auc_score(y_true, y_probs)
    except ValueError:
        auc = 0.0 # 如果验证集只有一类，那你的标注两周就白忙活了

    # 计算整体准确率
    acc = (y_preds == y_true).mean()

    print(f"Confusion Matrix:\n{cm}")
    print(f"ROC-AUC: {auc:.4f} | Accuracy: {acc:.4f}")

    return total_loss / n, acc, cm, auc

In [63]:
model.load_state_dict(torch.load("best.pt", map_location=device))
test_loss, test_acc,  cm, auc = evaluate(test_loader, model, criterion, device)
print(f"TEST: loss {test_loss:.3f}, acc {test_acc:.4f}")

Confusion Matrix:
[[0.9910306  0.0089694 ]
 [0.12755102 0.87244898]]
ROC-AUC: 0.9839 | Accuracy: 0.9871
TEST: loss 0.077, acc 0.9871
